In [ ]:
import pandas as pd
import numpy as np
import os



target_col = "Dropout"

df = pd.read_csv("data/balanced_dropout_dataset.csv")
print(df.shape)
df.head()

In [ ]:
numeric_cols = df.select_dtypes(include=[np.number]).columns.tolist()
categorical_cols = df.select_dtypes(include=['object', 'category']).columns.tolist()

if target_col in numeric_cols:
    numeric_cols.remove(target_col)
if target_col in categorical_cols:
    categorical_cols.remove(target_col)

print("Numeric:", numeric_cols)
print("Categorical:", categorical_cols)

In [ ]:
from sklearn.preprocessing import LabelEncoder

le_target = LabelEncoder()
y_encoded = le_target.fit_transform(df[target_col])

corr_with_target = df[numeric_cols].apply(lambda col: col.corr(pd.Series(y_encoded)))
corr_with_target = corr_with_target.sort_values(key=abs, ascending=False)
print(corr_with_target)

In [ ]:
from sklearn.feature_selection import mutual_info_classif

X_numeric = df[numeric_cols].fillna(df[numeric_cols].median())
mi_scores = mutual_info_classif(X_numeric, y_encoded, random_state=42)
mi_series = pd.Series(mi_scores, index=numeric_cols).sort_values(ascending=False)
print(mi_series)

In [ ]:
from scipy.stats import chi2_contingency

def cramers_v(x, y):
    confusion_matrix = pd.crosstab(x, y)
    chi2 = chi2_contingency(confusion_matrix)[0]
    n = confusion_matrix.sum().sum()
    phi2 = chi2 / n
    r, k = confusion_matrix.shape
    return np.sqrt(phi2 / min(k - 1, r - 1))

cat_scores = {}
for col in categorical_cols:
    cat_scores[col] = cramers_v(df[col].astype(str), df[target_col].astype(str))

cat_scores_series = pd.Series(cat_scores).sort_values(ascending=False)
print(cat_scores_series)

In [ ]:

CORR_THRESHOLD = 0.05
MI_THRESHOLD = 0.01
CRAMERS_V_THRESHOLD = 0.05

relevant_numeric = corr_with_target[
    (corr_with_target.abs() >= CORR_THRESHOLD) | (mi_series >= MI_THRESHOLD)
].index.tolist()

relevant_categorical = cat_scores_series[
    cat_scores_series >= CRAMERS_V_THRESHOLD
].index.tolist()

relevant_cols = relevant_numeric + relevant_categorical + [target_col]
print("Relevant columns kept:", relevant_cols)
print("Dropped columns:", [c for c in df.columns if c not in relevant_cols])

df_relevant = df[relevant_cols]
df_relevant.shape

In [ ]:
os.makedirs("data", exist_ok=True)
df_relevant.to_csv("data/relevant_cols.csv", index=False)
print("Saved to data/relevant_cols.csv")

In [ ]:
from sklearn.model_selection import train_test_split

df_model = df_relevant.copy()

for col in relevant_categorical:
    le = LabelEncoder()
    df_model[col] = le.fit_transform(df_model[col].astype(str))

X = df_model.drop(columns=[target_col])
y = LabelEncoder().fit_transform(df_model[target_col])

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

print(X_train.shape, X_test.shape)

In [ ]:
from xgboost import XGBClassifier

model = XGBClassifier(
    n_estimators=200,
    max_depth=5,
    learning_rate=0.1,
    eval_metric="logloss",
    random_state=42
)

model.fit(X_train, y_train)

In [ ]:
from sklearn.metrics import classification_report, confusion_matrix, roc_auc_score
import matplotlib.pyplot as plt
import seaborn as sns

y_pred = model.predict(X_test)
y_proba = model.predict_proba(X_test)[:, 1]

print(classification_report(y_test, y_pred))
print("ROC-AUC:", roc_auc_score(y_test, y_proba))

cm = confusion_matrix(y_test, y_pred)
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues')
plt.xlabel("Predicted")
plt.ylabel("Actual")
plt.title("Confusion Matrix")
plt.show()

In [ ]:
importances = pd.Series(model.feature_importances_, index=X.columns).sort_values(ascending=False)

plt.figure(figsize=(8, 6))
sns.barplot(x=importances.values, y=importances.index)
plt.title("XGBoost Feature Importances")
plt.show()

importances

In [ ]:
from sklearn.model_selection import RandomizedSearchCV

param_dist = {
    "n_estimators": [100, 200, 300, 500],
    "max_depth": [3, 4, 5, 6, 8],
    "learning_rate": [0.01, 0.05, 0.1, 0.2],
    "subsample": [0.6, 0.8, 1.0],
    "colsample_bytree": [0.6, 0.8, 1.0],
    "min_child_weight": [1, 3, 5],
}

search = RandomizedSearchCV(
    XGBClassifier(eval_metric="logloss", random_state=42),
    param_distributions=param_dist,
    n_iter=30,
    scoring="roc_auc",
    cv=5,
    random_state=42,
    n_jobs=-1
)

search.fit(X_train, y_train)
print("Best params:", search.best_params_)
print("Best AUC:", search.best_score_)

best_model = search.best_estimator_

In [ ]:
best_model = XGBClassifier(
    subsample=1.0,
    n_estimators=100,
    min_child_weight=5,
    max_depth=3,
    learning_rate=0.05,
    colsample_bytree=0.8,
    eval_metric="logloss",
    random_state=42
)

best_model.fit(X_train, y_train)

y_proba_best = best_model.predict_proba(X_test)[:, 1]
y_pred_best = best_model.predict(X_test)

print(classification_report(y_test, y_pred_best))
print("ROC-AUC:", roc_auc_score(y_test, y_proba_best))

In [ ]:
import pickle
import os
from datetime import datetime

os.makedirs("models", exist_ok=True)


model_name = f"xgb_dropout_relevant_cols.pkl"
model_path = os.path.join("models", model_name)

with open(model_path, "wb") as f:
    pickle.dump(best_model, f)

print(f"Model saved to: {model_path}")

In [ ]:
train_auc = roc_auc_score(y_train, best_model.predict_proba(X_train)[:, 1])
test_auc = roc_auc_score(y_test, y_proba_best)
print(f"Train AUC: {train_auc:.4f} | Test AUC: {test_auc:.4f}")

In [ ]:
import json

metadata = {
    "model_file": model_name,
    "algorithm": "XGBoost",
    "target": target_col,
    "features": list(X.columns),
    "best_threshold": float(best_threshold),
    "test_auc": float(test_auc),
    "train_auc": float(train_auc),
    "hyperparameters": best_model.get_params(),
    "date_trained": timestamp
}

metadata_path = os.path.join("models", f"xgb_dropout_relevant_cols.json")
with open(metadata_path, "w") as f:
    json.dump(metadata, f, indent=2)

print(f"Metadata saved to: {metadata_path}")